# Logistic Regression for Machine Failure Prediction

This notebook evaluates Logistic Regression for predicting machine failure
using three different class-imbalance handling strategies.

## Experiments

1. Logistic Regression without balancing
2. Logistic Regression with class weighting
3. Logistic Regression with SMOTE

## Evaluation Process

For each experiment:

- Fit the initial model
- Perform Stratified K-Fold Cross-Validation
- Perform hyperparameter tuning using GridSearchCV
- Select the best model based on PR-AUC
- Evaluate the best tuned model on the untouched test set

## Evaluation Metrics

- Precision
- Recall
- F1 Score
- ROC-AUC
- PR-AUC
- Confusion Matrix

The final comparison table is created only after all three experiments
have been completed.

In [4]:
# IMPORT REQUIRED LIBRARIES


import os
import sys
import joblib
import numpy as np
import pandas as pd

# Scikit-learn
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Imbalanced-learn
import imblearn

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# VERIFY ENVIRONMENT

print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Scikit-learn:", sklearn.__version__)
print("Imbalanced-learn:", imblearn.__version__)

print("\nAll libraries imported successfully.")

Python: /Users/sethukkarasipalani/Documents/AI4I-Predictive-Maintenance/.venv/bin/python
Python version: 3.11.16
Scikit-learn: 1.9.0
Imbalanced-learn: 0.14.2

All libraries imported successfully.


In [5]:
# LOAD DATA


DATA_PATH = "../data/processed/data_split"

X_train = pd.read_csv(
    f"{DATA_PATH}/X_train_scaled.csv"
)

X_test = pd.read_csv(
    f"{DATA_PATH}/X_test_scaled.csv"
)

y_train = pd.read_csv(
    f"{DATA_PATH}/y_train.csv"
).squeeze()

y_test = pd.read_csv(
    f"{DATA_PATH}/y_test.csv"
).squeeze()

# DISPLAY DATASET INFORMATION

print("Data loaded successfully.\n")

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

Data loaded successfully.

X_train shape: (8000, 9)
X_test shape : (2000, 9)
y_train shape: (8000,)
y_test shape : (2000,)


In [6]:
# CHECK TRAINING CLASS DISTRIBUTION

print("Training Class Distribution")
print("-" * 40)

train_counts = y_train.value_counts().sort_index()
train_percent = (
    y_train.value_counts(normalize=True)
    .sort_index() * 100
)

for class_value in train_counts.index:
    print(
        f"Class {class_value}: "
        f"{train_counts[class_value]} samples "
        f"({train_percent[class_value]:.2f}%)"
    )

# CHECK TEST CLASS DISTRIBUTION

print("\nTest Class Distribution")
print("-" * 40)

test_counts = y_test.value_counts().sort_index()
test_percent = (
    y_test.value_counts(normalize=True)
    .sort_index() * 100
)

for class_value in test_counts.index:
    print(
        f"Class {class_value}: "
        f"{test_counts[class_value]} samples "
        f"({test_percent[class_value]:.2f}%)"
    )

Training Class Distribution
----------------------------------------
Class 0: 7729 samples (96.61%)
Class 1: 271 samples (3.39%)

Test Class Distribution
----------------------------------------
Class 0: 1932 samples (96.60%)
Class 1: 68 samples (3.40%)


In [7]:
# STRATIFIED K-FOLD CROSS-VALIDATION

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Stratified Cross-Validation configured.")

5-Fold Stratified Cross-Validation configured.


In [8]:
# MULTI-METRIC SCORING

scoring = {
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),

    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),

    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),

    "roc_auc": "roc_auc",

    "pr_auc": "average_precision"
}

print("Scoring metrics configured successfully.")

Scoring metrics configured successfully.


In [9]:
# TEST SET EVALUATION FUNCTION

def evaluate_model(model, X_test, y_test, model_name):
    
    # Generate class predictions
    y_pred = model.predict(X_test)
    
    # Generate probability of class 1 (machine failure)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    
    # ========================================
    # CALCULATE METRICS
    # ========================================
    
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
    
    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
    
    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )
    
    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )
    
    pr_auc = average_precision_score(
        y_test,
        y_prob
    )
    
    cm = confusion_matrix(
        y_test,
        y_pred
    )
    
    # DISPLAY RESULTS

    
    print("=" * 65)
    print(model_name)
    print("=" * 65)
    
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")
    
    print("\nConfusion Matrix:")
    print(cm)
    
    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )
    

    # RETURN RESULTS
    
    return {
        "model": model_name,
        "test_precision": precision,
        "test_recall": recall,
        "test_f1": f1,
        "test_roc_auc": roc_auc,
        "test_pr_auc": pr_auc
    }

In [10]:
# ============================================
# EXPERIMENT 1
# LOGISTIC REGRESSION WITHOUT BALANCING
# ============================================

lr_no_balance = Pipeline([
    (
        "model",
        LogisticRegression(
            solver="liblinear",
            max_iter=2000,
            random_state=42
        )
    )
])

print("Logistic Regression model created.")

Logistic Regression model created.


In [11]:
# ============================================
# INITIAL MODEL FIT
# ============================================

lr_no_balance.fit(
    X_train,
    y_train
)

print("Initial model fitted successfully.")

Initial model fitted successfully.


In [12]:
# ============================================
# CROSS-VALIDATION
# ============================================

cv_no_balance = cross_validate(
    lr_no_balance,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)


# ============================================
# DISPLAY CV RESULTS
# ============================================

print("Experiment 1 - Initial Cross-Validation")
print("-" * 50)

print(
    f"Precision : "
    f"{cv_no_balance['test_precision'].mean():.4f} "
    f"+/- {cv_no_balance['test_precision'].std():.4f}"
)

print(
    f"Recall    : "
    f"{cv_no_balance['test_recall'].mean():.4f} "
    f"+/- {cv_no_balance['test_recall'].std():.4f}"
)

print(
    f"F1 Score  : "
    f"{cv_no_balance['test_f1'].mean():.4f} "
    f"+/- {cv_no_balance['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC   : "
    f"{cv_no_balance['test_roc_auc'].mean():.4f} "
    f"+/- {cv_no_balance['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC    : "
    f"{cv_no_balance['test_pr_auc'].mean():.4f} "
    f"+/- {cv_no_balance['test_pr_auc'].std():.4f}"
)

Experiment 1 - Initial Cross-Validation
--------------------------------------------------
Precision : 0.7773 +/- 0.0889
Recall    : 0.2436 +/- 0.0303
F1 Score  : 0.3707 +/- 0.0444
ROC-AUC   : 0.9137 +/- 0.0166
PR-AUC    : 0.4995 +/- 0.0298


In [13]:
# ============================================
# HYPERPARAMETER GRID
# ============================================

param_grid_lr = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


# ============================================
# GRID SEARCH
# ============================================

grid_no_balance = GridSearchCV(
    estimator=lr_no_balance,
    param_grid=param_grid_lr,
    cv=cv,
    scoring=scoring,
    refit="pr_auc",
    n_jobs=-1,
    return_train_score=True
)


# ============================================
# FIT GRID SEARCH
# ============================================

grid_no_balance.fit(
    X_train,
    y_train
)

print("Hyperparameter tuning completed.")
print("Best parameters:", grid_no_balance.best_params_)

Hyperparameter tuning completed.
Best parameters: {'model__C': 1}


In [14]:
# ============================================
# BEST CV RESULTS
# ============================================

best_index = grid_no_balance.best_index_

print("Experiment 1 - Best Tuned Model")
print("-" * 50)

print(
    "Best C:",
    grid_no_balance.best_params_["model__C"]
)

print(
    f"CV Precision : "
    f"{grid_no_balance.cv_results_['mean_test_precision'][best_index]:.4f}"
)

print(
    f"CV Recall    : "
    f"{grid_no_balance.cv_results_['mean_test_recall'][best_index]:.4f}"
)

print(
    f"CV F1        : "
    f"{grid_no_balance.cv_results_['mean_test_f1'][best_index]:.4f}"
)

print(
    f"CV ROC-AUC   : "
    f"{grid_no_balance.cv_results_['mean_test_roc_auc'][best_index]:.4f}"
)

print(
    f"CV PR-AUC    : "
    f"{grid_no_balance.cv_results_['mean_test_pr_auc'][best_index]:.4f}"
)

Experiment 1 - Best Tuned Model
--------------------------------------------------
Best C: 1
CV Precision : 0.7773
CV Recall    : 0.2436
CV F1        : 0.3707
CV ROC-AUC   : 0.9137
CV PR-AUC    : 0.4995


In [15]:
# ============================================
# GET BEST MODEL
# ============================================

best_lr_no_balance = grid_no_balance.best_estimator_


# ============================================
# FINAL TEST EVALUATION
# ============================================

result_no_balance = evaluate_model(
    best_lr_no_balance,
    X_test,
    y_test,
    "Logistic Regression - No Balancing"
)

Logistic Regression - No Balancing
Precision : 0.6000
Recall    : 0.1765
F1 Score  : 0.2727
ROC-AUC   : 0.9256
PR-AUC    : 0.4642

Confusion Matrix:
[[1924    8]
 [  56   12]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1932
           1       0.60      0.18      0.27        68

    accuracy                           0.97      2000
   macro avg       0.79      0.59      0.63      2000
weighted avg       0.96      0.97      0.96      2000



In [16]:
# ============================================
# EXPERIMENT 2
# LOGISTIC REGRESSION WITH CLASS WEIGHTING
# ============================================

lr_weight = Pipeline([
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            max_iter=2000,
            random_state=42
        )
    )
])

print("Class-weighted Logistic Regression created.")

Class-weighted Logistic Regression created.


In [17]:
# ============================================
# INITIAL MODEL FIT
# ============================================

lr_weight.fit(
    X_train,
    y_train
)

print("Initial class-weighted model fitted successfully.")

Initial class-weighted model fitted successfully.


In [18]:
# ============================================
# CROSS-VALIDATION
# ============================================

cv_weight = cross_validate(
    lr_weight,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)


# ============================================
# DISPLAY RESULTS
# ============================================

print("Experiment 2 - Initial Cross-Validation")
print("-" * 50)

print(
    f"Precision : "
    f"{cv_weight['test_precision'].mean():.4f} "
    f"+/- {cv_weight['test_precision'].std():.4f}"
)

print(
    f"Recall    : "
    f"{cv_weight['test_recall'].mean():.4f} "
    f"+/- {cv_weight['test_recall'].std():.4f}"
)

print(
    f"F1 Score  : "
    f"{cv_weight['test_f1'].mean():.4f} "
    f"+/- {cv_weight['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC   : "
    f"{cv_weight['test_roc_auc'].mean():.4f} "
    f"+/- {cv_weight['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC    : "
    f"{cv_weight['test_pr_auc'].mean():.4f} "
    f"+/- {cv_weight['test_pr_auc'].std():.4f}"
)

Experiment 2 - Initial Cross-Validation
--------------------------------------------------
Precision : 0.1624 +/- 0.0071
Recall    : 0.8489 +/- 0.0548
F1 Score  : 0.2726 +/- 0.0114
ROC-AUC   : 0.9235 +/- 0.0135
PR-AUC    : 0.4584 +/- 0.0405


In [19]:
# ============================================
# GRID SEARCH
# ============================================

grid_weight = GridSearchCV(
    estimator=lr_weight,
    param_grid=param_grid_lr,
    cv=cv,
    scoring=scoring,
    refit="pr_auc",
    n_jobs=-1,
    return_train_score=True
)


# ============================================
# FIT GRID SEARCH
# ============================================

grid_weight.fit(
    X_train,
    y_train
)

print("Hyperparameter tuning completed.")
print("Best parameters:", grid_weight.best_params_)

Hyperparameter tuning completed.
Best parameters: {'model__C': 0.1}


In [20]:
# ============================================
# BEST CV RESULTS
# ============================================

best_index = grid_weight.best_index_

print("Experiment 2 - Best Tuned Model")
print("-" * 50)

print(
    "Best C:",
    grid_weight.best_params_["model__C"]
)

print(
    f"CV Precision : "
    f"{grid_weight.cv_results_['mean_test_precision'][best_index]:.4f}"
)

print(
    f"CV Recall    : "
    f"{grid_weight.cv_results_['mean_test_recall'][best_index]:.4f}"
)

print(
    f"CV F1        : "
    f"{grid_weight.cv_results_['mean_test_f1'][best_index]:.4f}"
)

print(
    f"CV ROC-AUC   : "
    f"{grid_weight.cv_results_['mean_test_roc_auc'][best_index]:.4f}"
)

print(
    f"CV PR-AUC    : "
    f"{grid_weight.cv_results_['mean_test_pr_auc'][best_index]:.4f}"
)

Experiment 2 - Best Tuned Model
--------------------------------------------------
Best C: 0.1
CV Precision : 0.1572
CV Recall    : 0.8340
CV F1        : 0.2645
CV ROC-AUC   : 0.9163
CV PR-AUC    : 0.4868


In [21]:
# ============================================
# GET BEST MODEL
# ============================================

best_lr_weight = grid_weight.best_estimator_


# ============================================
# FINAL TEST EVALUATION
# ============================================

result_weight = evaluate_model(
    best_lr_weight,
    X_test,
    y_test,
    "Logistic Regression - Class Weighting"
)

Logistic Regression - Class Weighting
Precision : 0.1690
Recall    : 0.8824
F1 Score  : 0.2837
ROC-AUC   : 0.9320
PR-AUC    : 0.4483

Confusion Matrix:
[[1637  295]
 [   8   60]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.85      0.92      1932
           1       0.17      0.88      0.28        68

    accuracy                           0.85      2000
   macro avg       0.58      0.86      0.60      2000
weighted avg       0.97      0.85      0.89      2000



In [22]:
# ============================================
# EXPERIMENT 3
# SMOTE + LOGISTIC REGRESSION
# ============================================

lr_smote = ImbPipeline([
    (
        "smote",
        SMOTE(
            random_state=42
        )
    ),
    
    (
        "model",
        LogisticRegression(
            solver="liblinear",
            max_iter=2000,
            random_state=42
        )
    )
])

print("SMOTE + Logistic Regression pipeline created.")

SMOTE + Logistic Regression pipeline created.


In [23]:
# ============================================
# INITIAL MODEL FIT
# ============================================

lr_smote.fit(
    X_train,
    y_train
)

print("Initial SMOTE model fitted successfully.")

Initial SMOTE model fitted successfully.


In [24]:
# ============================================
# CROSS-VALIDATION
# ============================================

cv_smote = cross_validate(
    lr_smote,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)


# ============================================
# DISPLAY RESULTS
# ============================================

print("Experiment 3 - Initial Cross-Validation")
print("-" * 50)

print(
    f"Precision : "
    f"{cv_smote['test_precision'].mean():.4f} "
    f"+/- {cv_smote['test_precision'].std():.4f}"
)

print(
    f"Recall    : "
    f"{cv_smote['test_recall'].mean():.4f} "
    f"+/- {cv_smote['test_recall'].std():.4f}"
)

print(
    f"F1 Score  : "
    f"{cv_smote['test_f1'].mean():.4f} "
    f"+/- {cv_smote['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC   : "
    f"{cv_smote['test_roc_auc'].mean():.4f} "
    f"+/- {cv_smote['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC    : "
    f"{cv_smote['test_pr_auc'].mean():.4f} "
    f"+/- {cv_smote['test_pr_auc'].std():.4f}"
)

Experiment 3 - Initial Cross-Validation
--------------------------------------------------
Precision : 0.1645 +/- 0.0042
Recall    : 0.8083 +/- 0.0597
F1 Score  : 0.2732 +/- 0.0064
ROC-AUC   : 0.9185 +/- 0.0134
PR-AUC    : 0.4388 +/- 0.0459


In [25]:
# ============================================
# SMOTE HYPERPARAMETER GRID
# ============================================

param_grid_smote = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


# ============================================
# GRID SEARCH
# ============================================

grid_smote = GridSearchCV(
    estimator=lr_smote,
    param_grid=param_grid_smote,
    cv=cv,
    scoring=scoring,
    refit="pr_auc",
    n_jobs=-1,
    return_train_score=True
)


# ============================================
# FIT GRID SEARCH
# ============================================

grid_smote.fit(
    X_train,
    y_train
)

print("Hyperparameter tuning completed.")
print("Best parameters:", grid_smote.best_params_)

Hyperparameter tuning completed.
Best parameters: {'model__C': 0.1}


In [26]:
# ============================================
# BEST CV RESULTS
# ============================================

best_index = grid_smote.best_index_

print("Experiment 3 - Best Tuned Model")
print("-" * 50)

print(
    "Best C:",
    grid_smote.best_params_["model__C"]
)

print(
    f"CV Precision : "
    f"{grid_smote.cv_results_['mean_test_precision'][best_index]:.4f}"
)

print(
    f"CV Recall    : "
    f"{grid_smote.cv_results_['mean_test_recall'][best_index]:.4f}"
)

print(
    f"CV F1        : "
    f"{grid_smote.cv_results_['mean_test_f1'][best_index]:.4f}"
)

print(
    f"CV ROC-AUC   : "
    f"{grid_smote.cv_results_['mean_test_roc_auc'][best_index]:.4f}"
)

print(
    f"CV PR-AUC    : "
    f"{grid_smote.cv_results_['mean_test_pr_auc'][best_index]:.4f}"
)

Experiment 3 - Best Tuned Model
--------------------------------------------------
Best C: 0.1
CV Precision : 0.1633
CV Recall    : 0.8081
CV F1        : 0.2715
CV ROC-AUC   : 0.9139
CV PR-AUC    : 0.4688


In [27]:
# ============================================
# GET BEST MODEL
# ============================================

best_lr_smote = grid_smote.best_estimator_


# ============================================
# FINAL TEST EVALUATION
# ============================================

result_smote = evaluate_model(
    best_lr_smote,
    X_test,
    y_test,
    "Logistic Regression - SMOTE"
)

Logistic Regression - SMOTE
Precision : 0.1835
Recall    : 0.8824
F1 Score  : 0.3038
ROC-AUC   : 0.9310
PR-AUC    : 0.3902

Confusion Matrix:
[[1665  267]
 [   8   60]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.86      0.92      1932
           1       0.18      0.88      0.30        68

    accuracy                           0.86      2000
   macro avg       0.59      0.87      0.61      2000
weighted avg       0.97      0.86      0.90      2000



In [28]:
# ============================================
# COLLECT TEST RESULTS
# ============================================

results = [
    result_no_balance,
    result_weight,
    result_smote
]

comparison_df = pd.DataFrame(results)

comparison_df

,model,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,Logistic Regression - No Balancing,0.600000,0.176471,0.272727,0.925610,0.464225
1,Logistic Regression - Class Weighting,0.169014,0.882353,0.283688,0.932020,0.448345
2,Logistic Regression - SMOTE,0.183486,0.882353,0.303797,0.930954,0.390212
